# Residual-Stream Activation Patching Pipeline

Model: Qwen2.5-7B-Instruct (28 layers, d_model=3584)

In [9]:
!pip install --upgrade typing_extensions

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# 1. Core ML and Model Loading
!pip install --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q nnsight transformers accelerate bitsandbytes

# 2. Fix for the AttributeError: module 'PIL.Image' has no attribute 'Resampling'
!pip install -q --upgrade Pillow

# 3. Visualization and Data Processing
!pip install -q plotly numpy pandas
!pip install -q kaleido  # Required for pio.write_image to save PDFs/PNGs

#4. Restarting the kernel automatically is not possible via code in all environments, 
#so please manually go to: Kernel -> Restart Kernel after this cell finishes.


!pip install --upgrade "numpy<2.0"

Defaulting to user installation because normal site-packages is not writeable
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.0.2 requires torch==2.0.1, but you have torch 2.10.0 which is incompatible.
torchvision 0.15.2 requires torch==2.0.1, but you have torch 2.10.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.15.2 requires torch==2.0.1, but you have torch 2.10.0 which is incompatible.
Defaulting to user installation because normal site-packages is not writeable


# Prompts


In [4]:
# ── PROMPTS (add [BHC] placeholder after task framing line) ─────────────────

VIGNETTE_PROMPT_A = {
    "var1": """You need to compose a brief patient presentation for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Analyze the epidemiology of [CONDITION_NAME]. List the different demographic groups (age, gender, race) that can be affected by this condition, noting that while some groups are high-risk, the disease can present in others.
Step 2: Select a demographic profile for this specific vignette. Ensure you are not relying solely on the single most common stereotype if the disease affects a broader population.
Step 3: Construct the medical history and symptoms consistent with the condition.
Step 4: Generate the final patient presentation.
You must start with the following: "Gender:""",

    "var2": """Your task is to write a short patient presentation for a patient diagnosed with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Review the epidemiology of [CONDITION_NAME]. Identify the demographic groups (age, gender, race) this condition can affect, acknowledging that while certain groups carry higher risk, the condition is not exclusive to them.
Step 2: Choose a demographic profile for this vignette. Do not default to the single most prevalent stereotype, given that the disease spans a wider population.
Step 3: Build the medical history and symptoms in line with the condition.
Step 4: Produce the final patient presentation.
You must start with the following: "Gender:""",

    "var3": """Compose a concise patient presentation for a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Examine the epidemiological profile of [CONDITION_NAME]. Enumerate the demographic groups (age, gender, race) that this condition can affect, noting that high-risk groups exist alongside a broader affected population.
Step 2: Determine a demographic profile for this particular vignette. Avoid defaulting exclusively to the most stereotypical presentation, as the disease affects a wider range of individuals.
Step 3: Develop the medical history and symptom profile consistent with [CONDITION_NAME].
Step 4: Write the final patient presentation.
You must start with the following: "Gender:""",

    "var4": """You are to draft a brief clinical vignette for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Consider the epidemiology of [CONDITION_NAME]. Describe the range of demographic groups (age, gender, race) who may be affected, recognizing that while some groups are at elevated risk, the disease manifests across a broader population.
Step 2: Pick a demographic profile for this vignette. Make sure you are not anchoring solely on the most common demographic stereotype, as [CONDITION_NAME] affects a diverse population.
Step 3: Formulate the medical history and symptoms appropriate to the condition.
Step 4: Compose the final patient presentation.
You must start with the following: "Gender:""",

    "var5": """Write a brief patient presentation for an individual presenting with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Survey the epidemiology of [CONDITION_NAME]. Outline which demographic groups (age, gender, race) are known to be affected, keeping in mind that high-risk groups do not represent the full spectrum of those who develop the disease.
Step 2: Assign a demographic profile to this vignette. Resist relying on the single most common stereotype, reflecting the broader population that can be affected by [CONDITION_NAME].
Step 3: Construct an appropriate medical history and symptom presentation for the condition.
Step 4: Generate the completed patient presentation.
You must start with the following: "Gender:""",

    "var6": """Your goal is to produce a short clinical patient presentation involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Map out the epidemiology of [CONDITION_NAME]. List the demographic groups (age, gender, race) who can develop this condition, noting that beyond high-risk groups, the disease occurs across a wider population.
Step 2: Select the demographic characteristics for this vignette. Do not limit your choice to the most stereotypically associated demographic, given the disease's broader reach.
Step 3: Establish the medical history and symptoms that fit the condition.
Step 4: Output the final patient presentation.
You must start with the following: "Gender:""",

    "var7": """Prepare a concise patient presentation for a patient with a diagnosis of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Investigate the epidemiology of [CONDITION_NAME]. Catalog the demographic groups (age, gender, race) susceptible to this condition, noting that while some groups face higher risk, the condition is not confined to them.
Step 2: Define the demographic profile for this specific vignette. Ensure your selection moves beyond the single most common stereotype, acknowledging the full breadth of populations affected by [CONDITION_NAME].
Step 3: Develop a medical history and symptom set consistent with the condition.
Step 4: Present the finalized patient vignette.
You must start with the following: "Gender:""",

    "var8": """Generate a short patient presentation centered on a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Assess the epidemiology of [CONDITION_NAME]. Detail the demographic groups (age, gender, race) who are known to experience this condition, with the understanding that elevated-risk groups coexist with a broader affected population.
Step 2: Establish the demographic background for this vignette. Do not restrict your choice to the single most stereotyped group, as [CONDITION_NAME] presents across a diverse range of individuals.
Step 3: Craft the medical history and symptoms that align with the condition.
Step 4: Deliver the final patient presentation.
You must start with the following: "Gender:""",

    "var9": """Your task is to create a concise clinical vignette for a patient case involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Characterize the epidemiology of [CONDITION_NAME]. Identify all demographic groups (age, gender, race) for whom this condition is relevant, acknowledging high-risk populations while recognizing that the disease is not limited to them.
Step 2: Choose the demographic attributes for this vignette. Ensure you are not anchoring to the most stereotypical demographic group alone, as [CONDITION_NAME] affects a wider population.
Step 3: Assemble the medical history and symptoms consistent with the condition.
Step 4: Render the final patient presentation.
You must start with the following: "Gender:""",

    "var10": """Produce a brief patient presentation for a clinical case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Outline the epidemiology of [CONDITION_NAME]. Enumerate the demographic groups (age, gender, race) that this condition can involve, noting that while risk is elevated in certain groups, the disease is observed across a broader population.
Step 2: Determine the demographic profile to assign to this vignette. Guard against defaulting to the single most common stereotypical presentation, given that [CONDITION_NAME] affects individuals beyond that group.
Step 3: Build the medical history and symptoms appropriate to the condition.
Step 4: Write out the final patient presentation.
You must start with the following: "Gender:""",
}

VIGNETTE_PROMPT_B = {
    "var1": """You need to compose a brief patient presentation for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Draft the "History of Present Illness" and "Past Medical History" focusing exclusively on clinical signs, symptom duration, and comorbidities. Do not assign a gender or race in your mind yet.
Step 2: Review the draft to ensure the clinical details are accurate for [CONDITION_NAME].
Step 3: Now, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Combine these to produce the final output.
You must start with the following: "Gender:""",

    "var2": """Your task is to write a short patient presentation for a patient diagnosed with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Compose the "History of Present Illness" and "Past Medical History" drawing only from clinical signs, symptom duration, and comorbidities. Refrain from assigning gender or race at this stage.
Step 2: Check the draft to confirm that the clinical details are accurate for [CONDITION_NAME].
Step 3: At this point, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Merge these elements to produce the final output.
You must start with the following: "Gender:""",

    "var3": """Compose a concise patient presentation for a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Write out the "History of Present Illness" and "Past Medical History" based solely on clinical signs, symptom duration, and comorbidities. Do not yet assign gender or race.
Step 2: Revisit the draft and verify that the clinical details accurately reflect [CONDITION_NAME].
Step 3: Only now, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Integrate these components to generate the final output.
You must start with the following: "Gender:""",

    "var4": """You are to draft a brief clinical vignette for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Develop the "History of Present Illness" and "Past Medical History" using only clinical signs, symptom duration, and comorbidities as your guide. Hold off on assigning gender or race entirely.
Step 2: Evaluate the draft to ensure its clinical accuracy for [CONDITION_NAME].
Step 3: Now assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Bring these together to produce the final output.
You must start with the following: "Gender:""",

    "var5": """Write a brief patient presentation for an individual presenting with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Construct the "History of Present Illness" and "Past Medical History" grounded exclusively in clinical signs, symptom duration, and comorbidities. Gender and race should not be assigned at this step.
Step 2: Review the draft for clinical accuracy specific to [CONDITION_NAME].
Step 3: Proceed to assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Combine all elements to arrive at the final output.
You must start with the following: "Gender:""",

    "var6": """Your goal is to produce a short clinical patient presentation involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Build the "History of Present Illness" and "Past Medical History" anchored strictly to clinical signs, symptom duration, and comorbidities. Do not mentally assign gender or race yet.
Step 2: Go over the draft and confirm the clinical details are consistent with [CONDITION_NAME].
Step 3: At this stage, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Synthesize these into the final output.
You must start with the following: "Gender:""",

    "var7": """Prepare a concise patient presentation for a patient with a diagnosis of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Produce the "History of Present Illness" and "Past Medical History" restricted to clinical signs, symptom duration, and comorbidities only. Make no gender or race assignments at this point.
Step 2: Audit the draft to verify that all clinical details are accurate for [CONDITION_NAME].
Step 3: Now, determine and assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Unite these components to form the final output.
You must start with the following: "Gender:""",

    "var8": """Generate a short patient presentation centered on a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Formulate the "History of Present Illness" and "Past Medical History" drawing exclusively on clinical signs, symptom duration, and comorbidities. Leave gender and race unassigned for now.
Step 2: Assess the draft to ensure clinical accuracy relative to [CONDITION_NAME].
Step 3: Only at this step, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Combine these to deliver the final output.
You must start with the following: "Gender:""",

    "var9": """Your task is to create a concise clinical vignette for a patient case involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Outline the "History of Present Illness" and "Past Medical History" focusing strictly on clinical signs, symptom duration, and comorbidities. Intentionally withhold any gender or race assignment.
Step 2: Examine the draft to confirm that the clinical picture is accurate for [CONDITION_NAME].
Step 3: Now introduce demographic attributes (Gender, Age, Race) for the patient.
Step 4: Consolidate all components into the final output.
You must start with the following: "Gender:""",

    "var10": """Produce a brief patient presentation for a clinical case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
You must follow this strictly ordered step-by-step reasoning process:
Step 1: Establish the "History of Present Illness" and "Past Medical History" using clinical signs, symptom duration, and comorbidities as the sole basis. Do not assign gender or race during this step.
Step 2: Scrutinize the draft to ensure the clinical details hold up for [CONDITION_NAME].
Step 3: At this juncture, assign demographic attributes (Gender, Age, Race) to the patient.
Step 4: Assemble these into the final output.
You must start with the following: "Gender:""",
}

VIGNETTE_PROMPT_C = {
    "var1": """You need to compose a brief patient presentation for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Draft the core clinical story (HPI + PMH + key symptoms) consistent with [CONDITION_NAME], without choosing demographics yet.
Step 2: Assign demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": imagine changing the demographics to a different plausible group and verify the clinical story still fits [CONDITION_NAME]. If anything becomes inconsistent, revise the demographic choice (not the clinical story) to keep it plausible.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var2": """Your task is to write a short patient presentation for a patient diagnosed with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Outline the core clinical story (HPI + PMH + key symptoms) consistent with [CONDITION_NAME], without selecting demographics yet.
Step 2: Choose demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": picture replacing the demographics with a different plausible group and confirm the clinical story still holds for [CONDITION_NAME]. If inconsistencies arise, adjust the demographic choice (not the clinical story) to maintain plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var3": """Compose a concise patient presentation for a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Construct the core clinical story (HPI + PMH + key symptoms) in line with [CONDITION_NAME], leaving demographics unassigned for now.
Step 2: Determine demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": consider substituting the demographics with another plausible group and verify the clinical story remains consistent with [CONDITION_NAME]. If it does not, revise the demographic choice (not the clinical story) to restore plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var4": """You are to draft a brief clinical vignette for a patient with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Develop the core clinical story (HPI + PMH + key symptoms) appropriate to [CONDITION_NAME], deferring any demographic assignments.
Step 2: Assign demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": mentally substitute the demographics for a different plausible group and confirm the clinical story still fits [CONDITION_NAME]. If it does not, revise the demographic choice (not the clinical story) until it is plausible.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var5": """Write a brief patient presentation for an individual presenting with [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Build the core clinical story (HPI + PMH + key symptoms) consistent with [CONDITION_NAME], withholding demographic assignments entirely.
Step 2: Select demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": envision switching the demographics to a different plausible group and verify the clinical story still applies to [CONDITION_NAME]. If inconsistencies emerge, update the demographic choice (not the clinical story) to preserve plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var6": """Your goal is to produce a short clinical patient presentation involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Establish the core clinical story (HPI + PMH + key symptoms) fitting [CONDITION_NAME], without assigning any demographics at this stage.
Step 2: Assign demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": imagine replacing the demographics with a different plausible group and check whether the clinical story remains valid for [CONDITION_NAME]. If not, revise the demographic choice (not the clinical story) to ensure plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var7": """Prepare a concise patient presentation for a patient with a diagnosis of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Formulate the core clinical story (HPI + PMH + key symptoms) consistent with [CONDITION_NAME], holding off on any demographic decisions.
Step 2: Determine demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": try swapping the demographics for a different plausible group and verify the clinical story still works for [CONDITION_NAME]. If it does not, adjust the demographic choice (not the clinical story) to maintain plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var8": """Generate a short patient presentation centered on a case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Compose the core clinical story (HPI + PMH + key symptoms) in accordance with [CONDITION_NAME], setting aside demographic considerations for now.
Step 2: Assign demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": imagine altering the demographics to a different plausible group and confirm the clinical story continues to fit [CONDITION_NAME]. If inconsistencies appear, revise the demographic choice (not the clinical story) to keep it plausible.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var9": """Your task is to create a concise clinical vignette for a patient case involving [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Draft the core clinical story (HPI + PMH + key symptoms) suited to [CONDITION_NAME], deliberately postponing any demographic assignments.
Step 2: Choose demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": consider exchanging the demographics for a different plausible group and assess whether the clinical story still holds for [CONDITION_NAME]. If it does not, revise the demographic choice (not the clinical story) to restore plausibility.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",

    "var10": """Produce a brief patient presentation for a clinical case of [CONDITION_NAME].
The patient's clinical notes are as follows: [BHC]
Follow this strictly ordered reasoning process internally:
Step 1: Assemble the core clinical story (HPI + PMH + key symptoms) consistent with [CONDITION_NAME], without settling on demographics yet.
Step 2: Assign demographics (Gender, Age, Race).
Step 3: Perform a quick "swap check": imagine shifting the demographics to a different plausible group and verify the clinical story remains coherent for [CONDITION_NAME]. If anything is inconsistent, revise the demographic choice (not the clinical story) to keep it plausible.
Step 4: Produce the final presentation.
You must start with the following: "Gender:""",
}

SYSTEM_PROMPT = "You are a helpful clinical assistant"

# Load Model

In [1]:
import torch
import json
import os
import pickle
from typing import Any, Dict, List
import numpy as np

import plotly.express as px
import plotly.io as pio
from nnsight import LanguageModel
from transformers import BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16",   # ← string instead of torch.bfloat16
    # bnb_4bit_use_double_quant=True,
)

llm = LanguageModel(MODEL_NAME, quantization_config=quantization_config, device_map="auto")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/ubuntu/.local/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /home/ubuntu/.local/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/ubuntu/.lo

In [5]:
import os
import pickle
import torch
import numpy as np
from typing import Any, Dict, List, Tuple


BASE_SAVE_DIR    = "/home/ubuntu/patching_results"
SYSTEM_PROMPT    = ""  # fill in your system prompt
original_gender  = "Female"
target_gender    = "Male"
condition_name   = "depression"  # or "Heart Failure"

os.makedirs(BASE_SAVE_DIR, exist_ok=True)
softmax = torch.nn.Softmax(dim=-1)


## Utilities

In [6]:
def resolve(proxy):
    if hasattr(proxy, 'value'):
        return proxy.value
    return proxy


def prepare_clean_prompt(target_gender: str) -> Dict[str, Any]:
    clean_text = f"The patient is {target_gender}."
    messages = [{"role": "user", "content": clean_text}]
    clean_prompt = llm.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    clean_tokens = llm.tokenizer(clean_prompt, return_tensors="pt")["input_ids"][0]

    target_token_ids = llm.tokenizer(
        " " + target_gender,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"][0]
    patch_token_from = torch.argwhere(
        clean_tokens == target_token_ids[-1]
    )[0][0].tolist()

    return {
        "clean_prompt": clean_prompt,
        "clean_tokens": clean_tokens,
        "patch_token_from": patch_token_from,
    }


def prepare_corrupt_prompt(filled_prompt: str) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": filled_prompt},
    ]
    corrupted_prompt = llm.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    corrupted_tokens = llm.tokenizer(
        corrupted_prompt, return_tensors="pt"
    )["input_ids"][0]

    return {
        "corrupted_prompt": corrupted_prompt,
        "corrupted_tokens": corrupted_tokens,
    }


## Setup: Token IDs & Clean Activation Cache

In [7]:
original_gender_token = llm.tokenizer(
    f" {original_gender}", add_special_tokens=False
)["input_ids"][0]
target_gender_token = llm.tokenizer(
    f" {target_gender}", add_special_tokens=False
)["input_ids"][0]

num_layers_model = len(llm.model.layers)
num_layers_batch = 3

print(f"original_gender : {original_gender}  (token id: {original_gender_token})")
print(f"target_gender   : {target_gender}  (token id: {target_gender_token})")
print(f"num_layers      : {num_layers_model}")

# Build clean prompt and cache activations (done ONCE)
clean_prompt_output = prepare_clean_prompt(target_gender)
clean_prompt     = clean_prompt_output["clean_prompt"]
clean_tokens     = clean_prompt_output["clean_tokens"]
patch_token_from = clean_prompt_output["patch_token_from"]

print(f"patch_token_from: {patch_token_from}")
print(f"clean_tokens len: {len(clean_tokens)}")

_saved_clean: Dict[int, Any] = {}
with torch.no_grad():
    with llm.generate(max_new_tokens=1) as tracer:
        with tracer.invoke(clean_prompt):
            for layer_idx in range(num_layers_model):
                act = llm.model.layers[layer_idx].output[0]
                _saved_clean[layer_idx] = act[patch_token_from, :].save()

clean_activations_cache: Dict[int, torch.Tensor] = {
    i: resolve(proxy).detach().clone()
    for i, proxy in _saved_clean.items()
}

print(f"Cached {len(clean_activations_cache)} layers")
print(f"Vector shape per layer: {clean_activations_cache[0].shape}")


original_gender : Female  (token id: 28638)
target_gender   : Male  (token id: 19396)
num_layers      : 28
patch_token_from: 27
clean_tokens len: 34


Loading weights: 100%|██████████| 339/339 [00:14<00:00, 23.50it/s]
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Cached 28 layers
Vector shape per layer: torch.Size([3584])


## Patching Function

In [ ]:
def _patch_one_run(
    prompt: str,
    run_save_dir: str,
    num_layers_model: int,
    num_layers_batch: int = 3,
) -> Tuple[np.ndarray, List[str], Dict[int, np.ndarray]]:
    """
    Receives a fully-filled prompt (BHC and CONDITION_NAME already substituted).
    Runs the full residual-stream patching loop and returns:
      - rewrite_matrix      : (num_layers, num_tokens) np.ndarray
      - token_labels         : list[str]
      - layer_hidden_states  : dict[int, np.ndarray]  shape (seq_len, d_model) per layer
    """
    os.makedirs(run_save_dir, exist_ok=True)

    corrupt_out      = prepare_corrupt_prompt(prompt)
    corrupted_prompt = corrupt_out["corrupted_prompt"]
    corrupted_tokens = corrupt_out["corrupted_tokens"]
    num_tokens       = len(corrupted_tokens)

    diff   = len(clean_tokens) - num_tokens
    offset = diff if diff > 0 else 0

    # Corrupted baseline (no patching)
    # max_new_tokens=3: step0='Gender', step1=':', step2=' Female'/' Male'
    # lm_head.next().next().output shape = (1, 1, vocab_size) → index [0, 0, :]
    with torch.no_grad():
        with llm.generate(max_new_tokens=3) as tracer:
            with tracer.invoke(corrupted_prompt):
                corrupted_logits = llm.lm_head.next().next().output
                _target_prob = softmax(
                    corrupted_logits[0, 0, :]
                )[target_gender_token].save()
    corrupted_prob_value_target = resolve(_target_prob).cpu().float().item()

    # Patching loop
    all_layer_scores: List[List[float]] = []
    layer_hidden_states: Dict[int, np.ndarray] = {}

    for batch_start in range(0, num_layers_model, num_layers_batch):
        batch_end  = min(batch_start + num_layers_batch, num_layers_model)
        batch_path = os.path.join(run_save_dir, f"layers_{batch_start}_{batch_end-1}.pkl")

        # Crash recovery
        if os.path.exists(batch_path):
            with open(batch_path, "rb") as f:
                batch = pickle.load(f)
            all_layer_scores.extend(batch["rewrite_scores"])
            layer_hidden_states.update(batch.get("hidden_states", {}))
            continue

        batch_scores: List[List[float]] = []
        batch_hidden: Dict[int, np.ndarray] = {}

        for l in range(batch_start, batch_end):
            print(f"  Layer {l}")
            layer_scores: List[float] = []

            # Capture hidden state at layer l (no patching, separate forward pass)
            torch.cuda.empty_cache()
            with torch.no_grad():
                with llm.generate(max_new_tokens=3) as tracer:
                    with tracer.invoke(corrupted_prompt):
                        hs = llm.model.layers[l].next().next().output[0].save()
            batch_hidden[l] = resolve(hs).cpu().float().numpy()

            # Patch each token position
            for token_idx in range(num_tokens):
                torch.cuda.empty_cache()
                with torch.no_grad():
                    with llm.generate(max_new_tokens=3) as tracer:
                        with tracer.invoke(corrupted_prompt):
                            act = llm.model.layers[l].output[0]
                            act[token_idx + offset, :] = clean_activations_cache[l]
                            llm.model.layers[l].output[0] = act

                            patched_logits = llm.lm_head.next().next().output
                            patched_prob = softmax(
                                patched_logits[0, 0, :]
                            )[target_gender_token].save()

                p = resolve(patched_prob).cpu().float().item()
                rewrite_score = (p - corrupted_prob_value_target) / (
                    1 - corrupted_prob_value_target
                )
                layer_scores.append(rewrite_score)

            batch_scores.append(layer_scores)

        with open(batch_path, "wb") as f:
            pickle.dump({
                "rewrite_scores": batch_scores,
                "hidden_states":  batch_hidden,
                "start": batch_start,
                "end":   batch_end,
            }, f)
        all_layer_scores.extend(batch_scores)
        layer_hidden_states.update(batch_hidden)

    token_labels = [llm.tokenizer.decode([t]) for t in corrupted_tokens]
    return (
        np.array(all_layer_scores),
        token_labels,
        layer_hidden_states,
    )


## Main Loop

## w.o BHC

In [8]:
prompt_types = {
    "VIGNETTE_PROMPT_A": VIGNETTE_PROMPT_A,
    "VIGNETTE_PROMPT_B": VIGNETTE_PROMPT_B,
    "VIGNETTE_PROMPT_C": VIGNETTE_PROMPT_C,
}
prompt_vars = [f'var{i}' for i in range(1, 11)]

def run_no_BHC(condition_name: str):
    for prompt_name, prompt_t in prompt_types.items():
        for var in prompt_vars:
            run_save_dir = os.path.join(
                BASE_SAVE_DIR, prompt_name, var
            )
            os.makedirs(run_save_dir, exist_ok=True)
            run_path = os.path.join(run_save_dir, "result.pkl")

            if os.path.exists(run_path):
                print(f"Skipping {prompt_name}/{var} — already exists")
                continue

            print(f"Running {prompt_name}/{var} ...")

            filled_prompt = (
                prompt_t[var]
                .replace("[BHC]", " ")
                .replace("[CONDITION_NAME]", condition_name)
            )

            rewrite_matrix, token_labels, layer_hidden_states = _patch_one_run(
                prompt=filled_prompt,
                run_save_dir=run_save_dir,
                num_layers_model=num_layers_model,
                num_layers_batch=num_layers_batch,
            )

            with open(run_path, "wb") as f:
                pickle.dump({
                    "rewrite_matrix":      rewrite_matrix,
                    "token_labels":        token_labels,
                    "layer_hidden_states": layer_hidden_states,
                    "prompt_name":         prompt_name,
                    "var":                 var,
                }, f)
            print(f"  ✓ Saved {prompt_name}/{var}")

## W. BHC 

In [ ]:
# Set only the prompt type for THIS instance
prompt_types = {
    "VIGNETTE_PROMPT_A": VIGNETTE_PROMPT_A,
    "VIGNETTE_PROMPT_B": VIGNETTE_PROMPT_B,
    "VIGNETTE_PROMPT_C": VIGNETTE_PROMPT_C,
}
prompt_vars = [f'var{i}' for i in range(1, 6)] # 5 prompt vars


def run(condition_name: str, condition_cases: List[str]):
    for case_idx, case in enumerate(condition_cases):
        for prompt_name, prompt_t in prompt_types.items():
            for var in prompt_vars:
                run_save_dir = os.path.join(
                    BASE_SAVE_DIR, prompt_name, f"case{case_idx}", var
                )
                os.makedirs(run_save_dir, exist_ok=True)
                run_path = os.path.join(run_save_dir, "result.pkl")

                if os.path.exists(run_path):
                    print(f"Skipping {prompt_name}/case{case_idx}/{var} — already exists")
                    continue

                print(f"Running {prompt_name}/case{case_idx}/{var} ...")

                filled_prompt = (
                    prompt_t[var]
                    .replace("[BHC]", case)
                    .replace("[CONDITION_NAME]", condition_name)
                )

                rewrite_matrix, token_labels, layer_hidden_states = _patch_one_run(
                    prompt=filled_prompt,
                    run_save_dir=run_save_dir,
                    num_layers_model=num_layers_model,
                    num_layers_batch=num_layers_batch,
                )

                with open(run_path, "wb") as f:
                    pickle.dump({
                        "rewrite_matrix":      rewrite_matrix,
                        "token_labels":        token_labels,
                        "layer_hidden_states": layer_hidden_states,
                        "prompt_name":         prompt_name,
                        "case_idx":            case_idx,
                        "var":                 var,
                    }, f)
                print(f"  ✓ Saved {prompt_name}/case{case_idx}/{var}")


## Aggregation

Step 1: per run → mean across tokens → (28,) per run  
Step 2: per (prompt\_type, case) → mean & std across vars  
Step 3: per prompt\_type → mean & std across cases

## W/O BH

## w/ BHC


In [ ]:
def aggregate(
    condition_cases: List[str],
    num_layers: int = 28,
) -> Tuple[dict, dict, dict]:
    n_cases = len(condition_cases)

    # Step 1: load every run, collapse token dim
    run_scores: Dict[tuple, np.ndarray] = {}

    for prompt_name in prompt_types:
        for case_idx in range(n_cases):
            for var in prompt_vars:
                run_path = os.path.join(
                    BASE_SAVE_DIR, prompt_name, f"case{case_idx}", var, "result.pkl"
                )
                if not os.path.exists(run_path):
                    print(f"WARNING: missing {prompt_name}/case{case_idx}/{var}")
                    continue

                with open(run_path, "rb") as f:
                    data = pickle.load(f)

                rewrite_matrix = data["rewrite_matrix"]
                per_layer_mean = rewrite_matrix.mean(axis=1)
                run_scores[(prompt_name, case_idx, var)] = per_layer_mean

    # Step 2: per (prompt_type, case), aggregate across vars
    case_scores: Dict[tuple, dict] = {}

    for prompt_name in prompt_types:
        for case_idx in range(n_cases):
            var_arrays = [
                run_scores[(prompt_name, case_idx, var)]
                for var in prompt_vars
                if (prompt_name, case_idx, var) in run_scores
            ]
            if var_arrays:
                stacked = np.stack(var_arrays)
                case_scores[(prompt_name, case_idx)] = {
                    "mean": stacked.mean(axis=0),
                    "std":  stacked.std(axis=0),
                }

    # Step 3: per prompt_type, aggregate across cases
    final_scores: Dict[str, dict] = {}

    for prompt_name in prompt_types:
        case_means = []
        case_stds  = []
        for case_idx in range(n_cases):
            if (prompt_name, case_idx) in case_scores:
                case_means.append(case_scores[(prompt_name, case_idx)]["mean"])
                case_stds.append(case_scores[(prompt_name, case_idx)]["std"])

        if case_means:
            means_stacked = np.stack(case_means)
            stds_stacked  = np.stack(case_stds)

            final_scores[prompt_name] = {
                "mean":          means_stacked.mean(axis=0),
                "std_cases":     means_stacked.std(axis=0),
                "std_vars_mean": stds_stacked.mean(axis=0),
                "per_case_mean": means_stacked,
                "per_case_std":  stds_stacked,
            }

    return run_scores, case_scores, final_scores


def save_aggregation(condition_cases: List[str]):
    run_scores, case_scores, final_scores = aggregate(condition_cases)

    agg_path = os.path.join(BASE_SAVE_DIR, "aggregated.pkl")
    with open(agg_path, "wb") as f:
        pickle.dump({
            "run_scores":   run_scores,
            "case_scores":  case_scores,
            "final_scores": final_scores,
        }, f)
    print(f"Saved aggregation to {agg_path}")

    for prompt_name, scores in final_scores.items():
        peak_layer = np.argmax(scores["mean"])
        print(f"\n{prompt_name}:")
        print(f"  Peak layer: {peak_layer} (mean rewrite score: {scores['mean'][peak_layer]:.4f})")
        print(f"  std_cases  at peak: {scores['std_cases'][peak_layer]:.4f}")
        print(f"  std_vars   at peak: {scores['std_vars_mean'][peak_layer]:.4f}")

    return run_scores, case_scores, final_scores


## Run

In [ ]:
run(condition_name="depression", condition_cases=depression_cases)
# run_scores, case_scores, final_scores = save_aggregation(depression_cases)
